<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part3_transformers/08_vision_transformers/notebooks/08_vision_transformers_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 08: Vision Transformers

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

In 2020 a team at Google showed that the transformer, the architecture behind BERT and today's
large language models, also works for images if you cut each image into small square patches and
treat the patches like words. Vision transformers (ViTs) now sit inside most of the models in the
rest of this course: CLIP in module 09, the vision-language models in module 10, the detectors in
module 12.

In this notebook you will:

- cut an image into patches by hand, the way a ViT does
- open a pre-trained ViT and follow an image through it: tokens, layers, the class token and the
  position embeddings
- plot attention maps, and think about what they do and don't show
- fine-tune a ViT on the protest labels and score it on your 40 coded images
- compare it with a linear probe on frozen embeddings, and with everything on your scoreboard so far

**Data.** The protest and King's Day photos with your codes from module 04.

**Model.** `google/vit-base-patch16-224` (Apache 2.0): ViT-Base with 16 × 16 pixel patches,
pre-trained on ImageNet-21k (about 14 million images) and fine-tuned on the 1,000 ImageNet
categories.

**Runtime.** Use a GPU: *Runtime → Change runtime type → T4 GPU*. Sections 1 to 3 also run on a
CPU; fine-tuning in section 4 takes a few minutes on a T4 and much longer without one.

## 0. Setup

Everything we need is preinstalled in Colab. The course helpers load the data and the training
loop from module 05.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)
ON_GPU = DEVICE == "cuda"

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:28], fontsize=7)
    plt.tight_layout()
    plt.show()

train_table, test_table = course_data.protest_train_test()

## 1. An image as a sequence of patches

A text transformer reads a sequence of tokens. A ViT makes a sequence out of an image: it resizes
the image to 224 × 224 pixels and cuts it into a grid of 16 × 16 pixel patches, reading them left
to right and top to bottom like words on a page. Each patch is 16 × 16 pixels × 3 colors = 768
numbers. A single linear layer turns those 768 numbers into the patch's token embedding.

Let's do the cutting by hand. `unfold` slides a window over one dimension of a tensor; doing it
over height and width gives the grid.

In [ ]:
PATCH = 16
example_path = test_table["local_path"].iloc[0]
img = load_rgb(example_path, 1000).resize((224, 224))
x = torch.tensor(np.array(img)).permute(2, 0, 1).float() / 255        # (3, 224, 224)

grid = x.unfold(1, PATCH, PATCH).unfold(2, PATCH, PATCH)              # (3, 14, 14, 16, 16)
patches = grid.permute(1, 2, 0, 3, 4).reshape(-1, 3 * PATCH * PATCH)   # (196, 768)
print("grid:", tuple(grid.shape), "  patches as a sequence:", tuple(patches.shape))

n = grid.shape[1]
fig, axes = plt.subplots(n, n, figsize=(6, 6))
for i in range(n):
    for j in range(n):
        axes[i, j].imshow(grid[:, i, j].permute(1, 2, 0).numpy())
        axes[i, j].axis("off")
plt.suptitle("The same image as 196 patches, in reading order")
plt.subplots_adjust(wspace=0.08, hspace=0.08)
plt.show()

The model never sees the image as a whole again. From here on it works with 196 vectors, plus
one extra token at the front, the **class token** (`[CLS]`, as in BERT), which starts out the
same for every image and collects information from the patches as it moves through the layers.

In every layer, each token looks at every other token (**self-attention**). That is the big
difference from a CNN, whose filters only see a small neighborhood and need many layers before
information from opposite corners of the image meets. The price is that attention compares every
token with every other token, so the work grows with the *square* of the number of tokens.

> **✏️ Exercise 1**
>
> How many tokens (patches plus the class token) does a ViT process, and how many entries does one
> attention matrix have (tokens × tokens), for these settings?
>
> | image size | patch size | example |
> |---|---|---|
> | 224 | 16 | the model in this notebook |
> | 384 | 16 | ViT-B/16 fine-tuned at higher resolution |
> | 224 | 32 | ViT-B/32, the image encoder of CLIP in module 09 |
> | 224 | 14 | DINOv2 at the resolution we used in module 06 |
> | 518 | 14 | DINOv2 at its training resolution |
>
> What happens to the cost when you double the image size with the same patch size?

In [ ]:
# Your code here

## 2. Inside a pre-trained ViT

The image processor resizes the image to 224 × 224 and scales the pixel values the way the model
was trained. We load the model with `attn_implementation="eager"`: the faster attention code that
transformers uses by default doesn't return the attention weights, and we want to look at them.
`add_pooling_layer=False` leaves out a layer that this checkpoint doesn't have trained weights for.

In [ ]:
VIT = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(VIT)
vit = AutoModel.from_pretrained(VIT, add_pooling_layer=False, attn_implementation="eager").eval().to(DEVICE)

@torch.no_grad()
def run_vit(path):
    inputs = processor(images=load_rgb(path, 1000), return_tensors="pt").to(DEVICE)
    return vit(**inputs, output_hidden_states=True, output_attentions=True)

out = run_vit(example_path)
print("final tokens:     ", tuple(out.last_hidden_state.shape), "(images, tokens, numbers per token)")
print("hidden states:    ", len(out.hidden_states), "(the patch embeddings plus one per layer)")
print("attention weights:", len(out.attentions), "layers of", tuple(out.attentions[0].shape),
      "(images, heads, tokens, tokens)")

After the last layer, the class token is the summary of the whole image that the classification
head reads. It is also the usual **image embedding** of a ViT: `last_hidden_state[:, 0]`. (DINOv2
in module 06 returned its embedding as `pooler_output`, which for that model is the same class
token after the final normalization. Always check what "the embedding" means for the model you
use.)

**Position embeddings.** Self-attention by itself ignores order: shuffle the patches and the
result is the same. So before the first layer, every patch gets a learned vector added that says
where it sits. Nobody tells the model that patch 15 lies below patch 1; it has to learn that. We
can check whether it did by comparing the learned position embeddings with each other.

In [ ]:
pos = vit.embeddings.position_embeddings[0, 1:].detach().cpu()        # (196, hidden size), class token left out
pos = F.normalize(pos, dim=1)
similarity = (pos @ pos.T).numpy()
side = int(np.sqrt(len(pos)))

fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for ax, idx in zip(axes.flat, [0, 6, 13, 90, 97, 105, 182, 195]):
    ax.imshow(similarity[idx].reshape(side, side), cmap="viridis")
    r, c = divmod(idx, side)
    ax.scatter([c], [r], color="red", s=12)
    ax.set_title(f"patch {idx} (row {r}, col {c})", fontsize=8)
    ax.axis("off")
plt.suptitle("Similarity of each position embedding to the one marked in red")
plt.tight_layout()
plt.show()

Each map should light up around the red dot and along its row and column: the model has learned
a two-dimensional layout from nothing but the order of the patches. (If the maps look like noise,
you are probably running with random weights; check that the model loaded.)

> **✏️ Exercise 2**
>
> Tokens start out local: a patch embedding knows only its 16 × 16 pixels. As they pass through the
> layers they exchange information. For the example image, compute the average cosine similarity
> between all pairs of *patch* tokens (leave out the class token) after each layer, using
> `out.hidden_states`, and plot it against the layer number. What does the trend tell you?

In [ ]:
# Your code here

## 3. Attention maps, and what they don't tell you

Each layer has 12 attention **heads**, and each head decides, for every token, how much to take
from every other token. The row of the class token in the last layer says which patches fed most
directly into the final summary. Averaged over heads and laid out on the 14 × 14 grid, it gives
the familiar heat maps.

In [ ]:
def cls_attention(out, layer=-1):
    """How much the class token attends to each patch in one layer, averaged over heads."""
    att = out.attentions[layer][0].mean(dim=0)[0, 1:].float().cpu()    # (196,)
    side = int(np.sqrt(len(att)))
    return att.reshape(side, side).numpy()

def overlay(path, heat, ax, title=""):
    img = load_rgb(path, 1000).resize((224, 224))
    heat = np.array(plt.matplotlib.colors.Normalize()(heat))
    ax.imshow(img)
    ax.imshow(np.kron(heat, np.ones((16, 16))), cmap="inferno", alpha=0.55)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

examples = test_table["local_path"].head(4).tolist()
fig, axes = plt.subplots(2, 4, figsize=(10, 5.4))
for k, path in enumerate(examples):
    o = run_vit(path)
    axes[0, k].imshow(load_rgb(path, 1000).resize((224, 224)))
    axes[0, k].axis("off")
    overlay(path, cls_attention(o), axes[1, k], "last layer, class token")
plt.tight_layout()
plt.show()

A single layer's attention is only part of the story, because what a token passes on in layer 12
is already a mixture of what it gathered in layers 1 to 11. **Attention rollout** (Abnar &
Zuidema, 2020) tries to follow information through all layers: average the heads, add the
identity matrix to account for the residual connection that carries each token forward unchanged,
normalize each row to sum to one, and multiply the matrices of all layers together.

> **✏️ Exercise 3**
>
> Write a function `rollout(out)` that implements attention rollout as described above and returns
> the class token's row (without the class token itself) as a 14 × 14 map. Plot it next to the
> last-layer map for the four example images. Where do the two disagree?

In [ ]:
# Your code here

**Be careful with attention maps.** They are attractive, and they often highlight something
plausible, but attention weights say how information is *mixed*, not why the model decided what it
decided. The 12 heads in a layer often disagree with each other, averaging them is a choice, and
rollout adds its own assumptions. Large ViTs also produce a few tokens with unusually high
attention in empty background areas, which the model uses as scratch space; Darcet and colleagues
(2024) traced these artifacts and proposed extra "register" tokens to absorb them. And a map that
lights up the protest sign doesn't show that the sign *caused* the prediction. To test that, you
change the input (cover the sign) and see whether the output changes.

> **✏️ Exercise 4**
>
> A colleague trained a ViT to detect protest photos. For the paper, they want to include attention
> maps showing that "the model focuses on protest signs and banners". What would you advise them to
> do instead of, or in addition to, the maps?

*Your answer here*

## 4. Fine-tuning a ViT on the protest labels

Now the real test: does the newer architecture help on our task? We replace the 1,000-category
ImageNet head with a fresh two-category head ("other" and "protest") and train the whole network
on the protest training images, with the same split and early stopping as in modules 05 and 06.
transformers will warn that some weights were newly initialized: that is the new head, as intended.

Two details matter for transformers: a **small learning rate** (2e-5 rather than 1e-3), so the
pre-trained weights are adjusted instead of overwritten, and the **same preprocessing** the model
was trained with (224 × 224, pixel values scaled with the processor's mean and standard deviation).

In [ ]:
mean, std = processor.image_mean, processor.image_std
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean, std)])

fit_table, val_table = train_test_split(train_table, test_size=0.2, stratify=train_table["category_label"],
                                        random_state=SEED)
fit_loader = course_torch.make_loader(course_torch.ImageTable(fit_table["local_path"], fit_table["category_label"], train_tf),
                                      shuffle=True, batch_size=16)
val_loader = course_torch.make_loader(course_torch.ImageTable(val_table["local_path"], val_table["category_label"], test_tf),
                                      batch_size=16)
test_loader = course_torch.make_loader(course_torch.ImageTable(test_table["local_path"], test_table["protest"], test_tf),
                                       batch_size=16)

counts = np.bincount(fit_table["category_label"], minlength=2)
class_weights = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32)

class ViTClassifier(nn.Module):
    """A thin wrapper so the module 05 training loop gets plain logits back."""
    def __init__(self, name=VIT):
        super().__init__()
        self.vit = AutoModelForImageClassification.from_pretrained(
            name, num_labels=2, id2label={0: "other", 1: "protest"}, label2id={"other": 0, "protest": 1},
            ignore_mismatched_sizes=True)

    def forward(self, x):
        return self.vit(pixel_values=x).logits

torch.manual_seed(SEED)
vit_classifier = ViTClassifier()
EPOCHS = 5 if ON_GPU else 2
history = course_torch.train_model(vit_classifier, fit_loader, val_loader, epochs=EPOCHS, lr=2e-5,
                                   class_weights=class_weights)
course_torch.plot_history(history, "ViT-B/16 fine-tuned on protest labels")

In [ ]:
test_pred, test_prob = course_torch.predict(vit_classifier, test_loader)
print(course_data.log_score("ViT-B/16 fine-tuned", test_table["protest"], test_pred, module="08"))

> **✏️ Exercise 5**
>
> Fine-tuning updates all 86 million weights. The cheaper alternative from module 06 is a **linear
> probe**: keep the model frozen, use its class token as the image embedding, and fit a logistic
> regression on top. Extract class-token embeddings from `vit` (the frozen model from section 2) for
> the training and test images, fit a probe on the training images, score it on your codes and log
> it as `"linear probe, ViT-B/16 embeddings"`. How does it compare with fine-tuning, and with the
> DINOv2 probe from module 06?

In [ ]:
# Your code here

## 5. Your scoreboard so far

Every method so far was scored on the same 40 images, against your own codes.

In [ ]:
scores = course_data.show_scores()
display(scores)

if len(scores):
    ax = scores.set_index("method")["balanced_accuracy"].sort_values().plot.barh(figsize=(6, 0.35 * len(scores) + 1))
    ax.axvline(0.5, color="gray", linestyle=":")
    ax.set_xlabel("balanced accuracy on your 40 coded images")
    plt.show()
course_data.keep_my_files()

Two warnings before reading too much into the ranking. First, 40 images is a small test set: a
balanced accuracy of 0.80 on 40 images has a 95% confidence interval of roughly ±0.13 (more if
one of your classes is small), so differences of a few points between methods are mostly noise. Second, every method was
trained on the Commons category labels, which are only an approximation of your codes (module 04
showed how often they disagree). A method can't be more accurate on your codes than those labels
allow, however good its architecture.

> **✏️ Exercise 6**
>
> You have about 300 labeled images for a new visual category in your own research. Based on this
> notebook and modules 05 and 06, would you start with a CNN or a ViT, trained from scratch,
> fine-tuned, or used as a frozen embedding? What would make you change your mind?

*Your answer here*